In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from scipy.spatial import distance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# Data read as torch.float32 scaled into the range [-1.0:1.0]
# Data Preparation
transform = transforms.Compose([
    transforms.ToTensor(),  # Transform to tensor into the range [0.0:1.0]
    #transforms.Normalize(mean=0., std=(1/255.)) # Change range to [0.0: 255.0], manual scaling produces better results
    #transforms.Normalize(mean=0., std=(1.)) # range [0.0:1.0]
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.60MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.19MB/s]


In [ ]:
TRAIN_SET_SIZE = 500
#TRAIN_SET_SIZE = 5000
#TRAIN_SET_SIZE = 6000
#TRAIN_SET_SIZE = 60000 # Full train set
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=TRAIN_SET_SIZE, shuffle=False)
batch = next(iter(train_loader))

train_samples_float = batch[0]
train_labels = batch[1]

In [ ]:
TEST_SET_SIZE = 500 # Full test set
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=TEST_SET_SIZE, shuffle=False)
batch = next(iter(test_loader))

test_samples_float = batch[0]
test_labels = batch[1]

In [ ]:
# Convert dataset to uint8
# Data as torch.float32 into the range [0.0:255.0]

scale_factor = 2
offset = scale_factor
train_samples = (train_samples_float * scale_factor).round().to(torch.uint8)
test_samples = (test_samples_float * scale_factor).round().to(torch.uint8)

#train_samples = train_samples_float.to(torch.uint8)
#test_samples = test_samples_float.to(torch.uint8)

In [ ]:
# Constants
DIMENSIONS = 8192
FEATURE_LEVELS = 255
FEATURES = 28 * 28
CLASSES = 10

In [ ]:
# Key-value encoding
def Encode(dimensions, image, levels_hvs_bin, x_hvs_bin, y_hvs_bin):
    encoded = torch.zeros(dimensions, dtype=torch.int32)

    for y in range(image.shape[1]):
        for x in range(image.shape[2]):
            encoded += levels_hvs_bin[image[0][y][x].item()] ^ x_hvs_bin[x] ^ y_hvs_bin[y]

    # Addition with a majority vote threshold
    image_size = image.shape[1] * image.shape[2]
    encoded_bin = torch.where(encoded > (image_size // 2), 1, 0)

    return encoded_bin.to(dtype=torch.int8), encoded

In [ ]:
# Seed generation

def BinaryTensor(n, size):

    bin = torch.zeros(size, dtype=torch.uint8)

    for i in range(size):
        bin[size - 1 - i] = n % 2
        n //= 2

    return bin

def MNIST_HVBit(t):
    a = torch.zeros_like(t)
    b = torch.zeros_like(t)
    c = torch.zeros_like(t)

    a[0] = t[0] ^ t[1]
    a[1] = t[2] ^ t[3]
    a[2] = t[4] ^ t[5]
    a[3] = t[6] ^ t[7]
    a[4] = t[8] ^ t[9]
    a[5] = t[10] ^ t[11]
    a[6] = t[12] ^ t[13]

    b[0] = a[0] ^ a[1]
    b[1] = a[2] ^ a[3]
    b[2] = a[4] ^ a[5]
    b[3] = a[6] ^ a[1]

    c[0] = b[0] ^ b[1]
    c[1] = b[2] ^ b[3]

    #return b[2], c[0], c[1]

    seed1 = a[4] ^ a[5]                 # b[2]
    seed2 = a[0] ^ a[1] ^ a[2] ^ a[3]   # c[0]
    seed3 = a[4] ^ a[5] ^ a[6] ^ a[1]   # c[1]

    return seed1, seed2, seed3

gen_seed = torch.zeros(DIMENSIONS, dtype=torch.uint8)

size = 16
gen_seed1 = torch.zeros(DIMENSIONS, dtype=torch.uint8)
gen_seed2 = torch.zeros(DIMENSIONS, dtype=torch.uint8)
gen_seed3 = torch.zeros(DIMENSIONS, dtype=torch.uint8)

for i in range(DIMENSIONS):
    f, x, y = MNIST_HVBit(BinaryTensor(i, size).roll( -(i % 16)) )

    gen_seed1[i] = f
    gen_seed2[i] = x
    gen_seed3[i] = y

In [ ]:
# GENERATED SEED

# Create hypervectors table: each value has an hypervector
level_hvs_bin = torch.zeros([FEATURE_LEVELS, DIMENSIONS], dtype=torch.uint8)

for level in range(FEATURE_LEVELS):
    level_hvs_bin[level] = gen_seed1.roll(level)

# Create hypervectors table: each pixel position has two hypervectors
x_hvs_bin = torch.zeros([28, DIMENSIONS], dtype=torch.uint8)
y_hvs_bin = torch.zeros([28, DIMENSIONS], dtype=torch.uint8)

for i in range(28):
    x_hvs_bin[i] = gen_seed2.roll(i)
    y_hvs_bin[i] = gen_seed3.roll(i)

In [ ]:
# Encode train set

encoded_train_set = []
encoded_train_set.append(torch.zeros([TRAIN_SET_SIZE, DIMENSIONS], dtype=torch.int8))
encoded_train_set.append(torch.zeros([TRAIN_SET_SIZE, 1], dtype=torch.int8))

encoded_train_set[1] = train_labels

for i, sample in enumerate(train_samples):
    encoded_train_set[0][i], _ = Encode(DIMENSIONS, sample, level_hvs_bin, x_hvs_bin, y_hvs_bin)

#torch.save(encoded_train_set[0], "train_60k_D10000.pt")
#encoded_train_set[0] = torch.load('/content/drive/MyDrive/ColabNotebooks/HDC/RealDeal/MNIST/train_60k_D10000.pt', weights_only = True)

train_set = {
    'samples' : encoded_train_set[0],
    'labels' : encoded_train_set[1]
}

In [ ]:
# Create class hypervectors using different amounts of train samples

train_sizes = [500]

class_hvs = torch.zeros([len(train_sizes), CLASSES, DIMENSIONS], dtype=torch.int32)
class_hvs_bin = torch.zeros([len(train_sizes), CLASSES, DIMENSIONS], dtype=torch.int8)

for i, size in enumerate(train_sizes):

    encoded_samples = train_set['samples'][:size]
    labels = train_set['labels'][:size]

    # Create hypervector classes
    # Standard centroid-based classifier: this is the most basic version of the classifier, where it adds each sample’s hypervector to its corresponding class
    for sample, label in zip(encoded_samples, labels.tolist()):
        class_hvs[i][label] += sample

    # Number of added samples in each class
    class_count = torch.bincount(labels)

    # Binarization: majority vote threshold
    for cl in range(CLASSES):
        class_hvs_bin[i][cl] = torch.where(class_hvs[i][cl] > (class_count[cl] // 2), 1, 0)

    print(f'class_hvs[{i}]: {CLASSES} class hypervectors created from {train_sizes[i]} train samples')

class_hvs[0]: 10 class hypervectors created from 500 train samples


In [ ]:
def Tensor2Int(t):

    value = 0

    for i in range(len(t)):
        value += t[i].item() * 2 ** i

    return value

In [ ]:
delta_h = []

for j in range(DIMENSIONS):
    max = 0
    min = 1000000
    for k in class_hvs_bin[0]:
        if k[j] > max:
            max = k[j]

        if k[j] < min:
            min = k[j]

    delta_h.append(max - min)

delta_h = torch.tensor(delta_h)

In [ ]:
sample = train_samples[5]

hv_bin, hv = Encode(DIMENSIONS, sample, level_hvs_bin, x_hvs_bin, y_hvs_bin)

top = 3200
top_idxs = delta_h.topk(top).indices

image_file = open(f'image.txt', "w+")
for level in sample.flatten():
    image_file.write(f'{level.item():02x}\n')
image_file.close()

indexes_file = open(f'indexes.txt', "w+")
for idx in top_idxs:
    indexes_file.write(f'{idx:04x}\n')
indexes_file.close()

class_hvs_bin_t = class_hvs_bin[0].T

indexes_file = open(f'class_hvs.txt', "w+")
for idx in top_idxs:
    indexes_file.write(f'{Tensor2Int(class_hvs_bin_t[idx]):08x}\n')
indexes_file.close()

In [ ]:
similarity = torch.zeros([CLASSES])

for cl in range(CLASSES):
    similarity[cl] = 1 - distance.hamming(class_hvs_bin[0][cl][top_idxs], hv_bin[top_idxs])

print(similarity * len(top_idxs))


similarity = torch.zeros([CLASSES])

for cl in range(CLASSES):
    for i in top_idxs:
        similarity[cl] += 1 if class_hvs_bin[0][cl][i] == hv_bin[i] else 0

print(similarity, similarity.argmin())


tensor([1926.0000, 1985.0000, 2251.0000, 1896.9999, 2041.0000, 2072.0000,
        2024.0000, 2020.0001, 2019.0000, 2115.0000])
tensor([1926., 1985., 2251., 1897., 2041., 2072., 2024., 2020., 2019., 2115.]) tensor(3)


In [ ]:
torch.set_printoptions(profile="full")
print(hv_bin[top_idxs][:10])
print(hv_bin[top_idxs][10:20])
print(hv_bin[top_idxs][20:30])
print(hv_bin[top_idxs][30:32])

#print(class_hvs_bin[0].shape)
#print(class_hvs_bin[0,:, top_idxs])


#class_hvs_bin_t = class_hvs_bin[0].T
#print(class_hvs_bin_t.shape)
#print(class_hvs_bin_t[top_idxs, :] )


tensor([1, 0, 0, 0, 1, 0, 0, 1, 1, 0], dtype=torch.int8)
tensor([0, 1, 1, 1, 0, 1, 1, 0, 1, 0], dtype=torch.int8)
tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 0], dtype=torch.int8)
tensor([1, 0], dtype=torch.int8)
